In [ ]:
import OptimusPrimus

from OptimusPrimus import OptimusPrimus

print("Libraries imported successfully.")

In [ ]:
IMAGE_SPEC = "ARCI URAS26F2"

SEG_ENCODER = "efficientnet-b7"
CLASS_MODEL_TYPE = "efficientnet_b0"

SEG_MODEL_SPEC = f"seg_model_{SEG_ENCODER}_{IMAGE_SPEC}"
CLASS_MODEL_SPEC = f"class_model_{CLASS_MODEL_TYPE}_{IMAGE_SPEC}"

IMAGE_CONFIG = {}
SEG_MODEL_CONFIG = {}
CLASS_MODEL_CONFIG = {}

print(f"Segmentation model spec  : {SEG_MODEL_SPEC}")
print(f"Classification model spec: {CLASS_MODEL_SPEC}")

In [ ]:
training_run = False

if training_run:

    from OptimusPrimus import OptimusPrimusTraining

    TRAIN_IMAGE_DIR = f"Data/images/{IMAGE_SPEC}"

    TRAIN_IMAGE = {}
    TRAIN_MODEL_CONFIG = {}
    TRAINING_PARAMETERS = {}

    train_track_detector = OptimusPrimusTraining(
        TRAIN_IMAGE_DIR,
        IMAGE_SPEC,
        train_image=TRAIN_IMAGE,
        train_model_config=TRAIN_MODEL_CONFIG,
        training_parameters=TRAINING_PARAMETERS,
        image_config=IMAGE_CONFIG,
        parallel=True,  # no GPU -> spread across all CPU cores; set False for single-core
    )

    train_track_detector.perform_seg_training()

    train_track_detector.perform_class_training(train_track_detector.seg_best_model_path, seg_th=0.1)

    dict_seg_efficiency = train_track_detector.evaluate_binned_efficiency(
        'seg',
        seg_th=0.1,
        cls_th=0.55,
    )

    dict_seg_class_efficiency = train_track_detector.evaluate_binned_efficiency(
        'seg_class',
        seg_th=0.1,
        cls_th=0.55,
    )

    print(dict_seg_efficiency['efficiency_table'])

In [ ]:
track_detector = OptimusPrimus(
    IMAGE_SPEC,
    SEG_MODEL_SPEC,
    CLASS_MODEL_SPEC,
    image_config=IMAGE_CONFIG,
    seg_model_config=SEG_MODEL_CONFIG,
    class_model_config=CLASS_MODEL_CONFIG,
    parallel=True,
)

print(f"Image tiles folder        : {track_detector.image_path}")
print(f"Segmentation model file   : {track_detector.seg_model_path}")
print(f"Classification model file : {track_detector.cls_model_path}")
print(f"Compute device            : {track_detector.device}")

In [ ]:
track_detector.perform_full_inference(seg_th=track_detector.seg_model_th, cls_th=track_detector.cls_model_th)

print(f"\nSegmentation-only tracks found      : {len(track_detector.seg_ellipses)}")
print(f"Segmentation + classification tracks: {len(track_detector.seg_cls_ellipses)}")
track_detector.seg_cls_ellipses.head()

In [ ]:
seg_stats = track_detector.get_track_distributions('seg')
seg_cls_stats = track_detector.get_track_distributions('seg_class')

seg_stats, seg_cls_stats

In [ ]:
print(f"Segmentation CSV          : {track_detector.seg_output_path}")
print(f"Segmentation+classification CSV: {track_detector.seg_cls_output_path}")

track_detector.inference_from_file(track_detector.seg_output_path)
track_detector.get_track_distributions('seg')

track_detector.inference_from_file(track_detector.seg_cls_output_path)
track_detector.get_track_distributions('seg_class')